# 02 — Build the FAISS index

**Purpose:** build and verify the job index interactively.

- Load and chunk everything in `data/jobs/`
- Embed the chunks and build the FAISS index
- Persist it to `vectorstore/jobs/`
- Reload it and confirm a test query returns sensible jobs

This notebook calls the same functions that back `python -m src.search.job_search --build`
and the admin "Build / rebuild job index" button in the Streamlit app — it exists to make
the process inspectable step by step.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.config import load_config
from src.search.job_search import load_job_dataset, build_job_index, search_jobs, _default_csv_path

config = load_config()
csv_path = _default_csv_path(config)
print('Job dataset:', csv_path)

In [ ]:
documents = load_job_dataset(csv_path)
print(f'Loaded {len(documents)} job postings.')
print()
print('Sample document:')
print(documents[0])

In [ ]:
# Build the index (embeds every job description locally, no API cost) and persist it.
index = build_job_index()
print(f'FAISS index built with {index.index.ntotal} vectors, '
      f'persisted to {config.vectorstore_dir / "jobs"}')

In [ ]:
# Reload from disk (exactly what the app does) and run a few test queries.
test_queries = [
    'SQL, Python, Pandas, Excel, Power BI, Statistics. Data analytics internship building dashboards.',
    'Python, PyTorch, deep learning, computer vision, model deployment.',
    'JavaScript, React, CSS, HTML, responsive design.',
    'Recruiting, sourcing, applicant tracking systems, technical screening.',
]

for q in test_queries:
    print('QUERY:', q[:70], '...')
    for r in search_jobs(q, top_k=3):
        print(f"   {r['score']:.3f}  {r['title']} @ {r['company']}")
    print()

### Observation

Each query's top results are semantically on-topic (data queries surface analyst/scientist
roles, ML queries surface ML/AI roles, frontend queries surface frontend roles) even though
none of the query text matches job titles verbatim — this is the semantic-search behaviour
the spec asks for ("find related jobs by meaning, not keywords"), as opposed to a keyword match
which would fail whenever the candidate's wording differs from the job posting's wording.